In [2]:
!pip -q install lightgbm

import os, json, joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
from google.colab import files

os.makedirs("data/raw", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

print("Upload: simulated_energy_data.csv, campus_rooms.csv, forecast_lgbm.joblib")
uploaded = files.upload()
for name in uploaded:
    dest = "models" if name.endswith(".joblib") else "data/raw"
    os.replace(name, f"{dest}/{name}")
print(os.listdir("data/raw"), os.listdir("models"))

Upload: simulated_energy_data.csv, campus_rooms.csv, forecast_lgbm.joblib


Saving forecast_lgbm.joblib to forecast_lgbm.joblib
['campus_rooms.csv', 'simulated_energy_data.csv'] ['forecast_lgbm.joblib']


In [3]:
DATA = "data/raw"
TARGET = "total_power_kw"

bundle = joblib.load("models/forecast_lgbm.joblib")
model, FEATURES, CATS = bundle["model"], bundle["features"], bundle["categories"]

df = pd.read_csv(f"{DATA}/simulated_energy_data.csv", parse_dates=["timestamp"])
rooms = pd.read_csv(f"{DATA}/campus_rooms.csv")[["room_id", "capacity"]]
df = (df.merge(rooms, on="room_id")
        .sort_values(["room_id", "timestamp"])
        .reset_index(drop=True))

# Same features used in notebook 01
g = df.groupby("room_id")[TARGET]
df["lag_24"] = g.shift(24)
df["lag_168"] = g.shift(168)
df["lag_336"] = g.shift(336)
df["same_hour_mean_7d"] = sum(g.shift(24 * k) for k in range(1, 8)) / 7
df["mean_prev_24h"] = g.transform(lambda s: s.shift(24).rolling(24).mean())
df["scheduled"] = df["scheduled"].astype(int)
df["sched_lag_168"] = df.groupby("room_id")["scheduled"].shift(168)
df["hour"] = df.timestamp.dt.hour
df["dow"] = df.timestamp.dt.dayofweek
for c, cats in CATS.items():
    df[c] = pd.Categorical(df[c], categories=cats)
df = df.dropna(subset=["lag_336"]).reset_index(drop=True)

# Keep only the period the model was NOT trained on (honest evaluation)
oos = df[df.timestamp >= "2026-10-18"].copy()
oos["pred_kw"] = model.predict(oos[FEATURES])

print(oos.shape, oos.timestamp.min(), "->", oos.timestamp.max())
oos[["timestamp", "room_id", TARGET, "pred_kw"]].head()

(20160, 25) 2026-10-18 00:00:00 -> 2026-11-21 23:00:00


,timestamp,room_id,total_power_kw,pred_kw
3192,2026-10-18 00:00:00,R01,0.298,0.292289
3193,2026-10-18 01:00:00,R01,0.289,0.286450
3194,2026-10-18 02:00:00,R01,0.291,0.291001
3195,2026-10-18 03:00:00,R01,0.282,0.286691
3196,2026-10-18 04:00:00,R01,0.278,0.293389


In [4]:
campus = (oos.groupby("timestamp")
             .agg(actual_kw=(TARGET, "sum"), pred_kw=("pred_kw", "sum"),
                  temp=("outside_temp_c", "first"), day_type=("day_type", "first"))
             .reset_index())
campus["date"] = campus.timestamp.dt.date

def daily_peak(col):
    return campus.loc[campus.groupby("date")[col].idxmax()].set_index("date")

act, prd = daily_peak("actual_kw"), daily_peak("pred_kw")
daily = pd.DataFrame({
    "day_type": act.day_type.astype(str),
    "actual_peak_kw": act.actual_kw.round(2),
    "pred_peak_kw": prd.pred_kw.round(2),
    "actual_peak_hour": act.timestamp.dt.hour,
    "pred_peak_hour": prd.timestamp.dt.hour,
})
daily["error_kw"] = (daily.pred_peak_kw - daily.actual_peak_kw).round(2)

# Predicted peak WINDOW = the 3 highest predicted hours of each day
top3 = (campus.sort_values(["date", "pred_kw"], ascending=[True, False])
               .groupby("date").head(3)
               .groupby("date").timestamp.apply(lambda s: set(s.dt.hour)))
daily["hit_top3_window"] = [h in top3[d] for d, h in zip(daily.index, daily.actual_peak_hour)]
daily["hour_diff"] = (daily.actual_peak_hour - daily.pred_peak_hour).abs()

work = daily[daily.day_type == "regular"]      # peaks on weekends are not meaningful
print("Working days:", len(work), "| all days:", len(daily))
print(f"Peak value MAE (kW): {work.error_kw.abs().mean():.2f}  (mean actual peak {work.actual_peak_kw.mean():.1f} kW)")
print(f"Peak value error (%): {(work.error_kw.abs() / work.actual_peak_kw).mean() * 100:.1f}")
print(f"Exact peak hour:      {(work.hour_diff == 0).mean() * 100:.0f}%")
print(f"Within 1 hour:        {(work.hour_diff <= 1).mean() * 100:.0f}%")
print(f"Actual peak inside predicted top-3 window: {work.hit_top3_window.mean() * 100:.0f}%")
work.head(10)

Working days: 25 | all days: 35
Peak value MAE (kW): 0.86  (mean actual peak 41.2 kW)
Peak value error (%): 2.1
Exact peak hour:      48%
Within 1 hour:        88%
Actual peak inside predicted top-3 window: 100%


,day_type,actual_peak_kw,pred_peak_kw,actual_peak_hour,pred_peak_hour,error_kw,hit_top3_window,hour_diff
date,,,,,,,,
2026-10-18,regular,47.20,45.38,9,9,-1.82,True,0
2026-10-19,regular,34.85,35.91,15,15,1.06,True,0
2026-10-20,regular,41.96,43.14,14,14,1.18,True,0
2026-10-21,regular,42.54,40.67,14,15,-1.87,True,1
2026-10-22,regular,45.26,45.22,13,13,-0.04,True,0
2026-10-25,regular,42.04,43.75,8,9,1.71,True,1
2026-10-26,regular,37.02,36.22,15,15,-0.80,True,0
2026-10-27,regular,43.71,43.92,14,15,0.21,True,1
2026-10-28,regular,44.28,43.44,14,15,-0.84,True,1


In [5]:
hourly = (oos.groupby("timestamp")
            .agg(actual_kw=(TARGET, "sum"), pred_kw=("pred_kw", "sum"), day_type=("day_type", "first"))
            .reset_index())
hourly["date"] = hourly.timestamp.dt.date

calib = hourly[hourly.timestamp < "2026-11-01"]            # used to learn the error pattern
test = hourly[hourly.timestamp >= "2026-11-01"].copy()     # used only for evaluation

busy = calib[calib.pred_kw > 15]                           # busy hours only
ratio = (busy.actual_kw / busy.pred_kw).values

print(f"Calibration hours: {len(ratio)}")
print("actual/predicted ratio quantiles (5%, 50%, 95%):", np.quantile(ratio, [0.05, 0.5, 0.95]).round(3))

Calibration hours: 88
actual/predicted ratio quantiles (5%, 50%, 95%): [0.914 0.996 1.052]


In [6]:
THRESHOLD_KW = 40   # the limit you want to monitor (change it freely)

# P(load > threshold) for each hour = forecast x every historical error ratio
test["prob_exceed"] = [(p * ratio > THRESHOLD_KW).mean() if p > 15 else 0.0 for p in test.pred_kw]
test["exceeded"] = test.actual_kw > THRESHOLD_KW
alert = test.prob_exceed >= 0.5

tp = int((alert & test.exceeded).sum())
fp = int((alert & ~test.exceeded).sum())
fn = int((~alert & test.exceeded).sum())
print(f"Threshold {THRESHOLD_KW} kW | hours that really exceeded: {int(test.exceeded.sum())}")
print(f"Alerts: TP={tp}  FP={fp}  FN={fn}")
print(f"Precision={tp / max(tp + fp, 1):.2f}  Recall={tp / max(tp + fn, 1):.2f}  "
      f"Brier={((test.prob_exceed - test.exceeded) ** 2).mean():.3f}")

# Daily summary: peak window + probability that the day exceeds the threshold
rows = []
for date, day in test.groupby("date"):
    hours = sorted(day.nlargest(3, "pred_kw").timestamp.dt.hour)
    rows.append({
        "date": date,
        "peak_window": ", ".join(f"{h:02d}:00" for h in hours),
        "pred_peak_kw": round(day.pred_kw.max(), 1),
        "actual_peak_kw": round(day.actual_kw.max(), 1),
        "P(exceed)": round(1 - np.prod(1 - day.prob_exceed), 2),
        "really_exceeded": bool(day.exceeded.any()),
    })
summary = pd.DataFrame(rows)
summary[summary.pred_peak_kw > 20]

Threshold 40 kW | hours that really exceeded: 18
Alerts: TP=15  FP=9  FN=3
Precision=0.62  Recall=0.83  Brier=0.016


,date,peak_window,pred_peak_kw,actual_peak_kw,P(exceed),really_exceeded
0,2026-11-01,"08:00, 09:00, 11:00",43.4,41.4,0.99,True
1,2026-11-02,"13:00, 14:00, 15:00",33.7,33.7,0.00,False
2,2026-11-03,"13:00, 14:00, 15:00",39.5,38.9,0.50,False
3,2026-11-04,"13:00, 14:00, 15:00",40.1,40.1,0.63,True
4,2026-11-05,"08:00, 09:00, 13:00",43.2,42.7,1.00,True
7,2026-11-08,"08:00, 09:00, 11:00",43.5,42.1,0.99,True
8,2026-11-09,"13:00, 14:00, 15:00",36.1,36.9,0.00,False
9,2026-11-10,"13:00, 14:00, 15:00",43.0,43.1,0.99,True
10,2026-11-11,"11:00, 14:00, 15:00",42.2,41.4,0.97,True
11,2026-11-12,"08:00, 09:00, 13:00",43.8,44.7,1.00,True


In [7]:
# Hours where we would raise an alert (probability >= 50%), test period only
alert_hours = test.loc[test.prob_exceed >= 0.5, "timestamp"]
rows_alert = oos[oos.timestamp.isin(alert_hours) & (oos.timestamp >= "2026-11-01")]
n_alert = rows_alert.timestamp.nunique()

contrib = (rows_alert.groupby(["building_id", "room_type"], observed=True).pred_kw.sum() / n_alert
           ).round(2).rename("avg_kw_in_alert_hours").reset_index()
contrib["share_%"] = (contrib.avg_kw_in_alert_hours / contrib.avg_kw_in_alert_hours.sum() * 100).round(1)

by_comp = rows_alert[["hvac_power_kw", "lighting_power_kw", "equipment_power_kw"]].sum()
print(f"Alert hours analysed: {n_alert}")
print("\nLoad share by component (%):")
print((by_comp / by_comp.sum() * 100).round(1).to_string())

contrib.sort_values("avg_kw_in_alert_hours", ascending=False)

Alert hours analysed: 24

Load share by component (%):
hvac_power_kw         50.3
lighting_power_kw     19.3
equipment_power_kw    30.4


,building_id,room_type,avg_kw_in_alert_hours,share_%
7,B03,Office,8.55,20.2
0,B01,Classroom,7.99,18.8
4,B02,Lab,6.48,15.3
3,B02,Classroom,6.39,15.1
1,B01,Lab,4.73,11.2
2,B01,Lecture Hall,4.01,9.5
5,B02,Lecture Hall,2.92,6.9
6,B03,Meeting Room,1.35,3.2


In [9]:
# Assumed rates (these are ASSUMPTIONS - change them and re-run)
RATES = {"hvac": 0.10, "lighting": 0.15, "lab_equipment_shift": 0.30}

r = rows_alert.copy()
sched = (r.scheduled == 1)
r["saved_hvac"] = RATES["hvac"] * r.hvac_power_kw * sched
r["saved_light"] = RATES["lighting"] * r.lighting_power_kw * sched
r["shifted_lab"] = np.where(r.room_type == "Lab", RATES["lab_equipment_shift"] * r.equipment_power_kw, 0.0)

eff = r.groupby("timestamp")[["saved_hvac", "saved_light", "shifted_lab"]].sum()

effect_cols = ["saved_hvac", "saved_light", "shifted_lab"]
sim = test.set_index("timestamp").join(eff)
sim[effect_cols] = sim[effect_cols].fillna(0)          # fill only the effect columns
sim["new_kw"] = sim.actual_kw - sim.saved_hvac - sim.saved_light - sim.shifted_lab

# Shifted lab-equipment energy is moved to 16:00 and 17:00 of the same day (off-peak hours)
shift_by_day = sim.groupby(sim.index.date).shifted_lab.sum()
for day, kwh in shift_by_day.items():
    for hour in (16, 17):
        ts = pd.Timestamp(day) + pd.Timedelta(hours=hour)
        if ts in sim.index:
            sim.loc[ts, "new_kw"] += kwh / 2

before = int((sim.actual_kw > THRESHOLD_KW).sum())
after = int((sim.new_kw > THRESHOLD_KW).sum())
print(f"Hours above {THRESHOLD_KW} kW: before={before}  after={after}")
print(f"Highest load: before={sim.actual_kw.max():.1f}  after={sim.new_kw.max():.1f} kW")
tot = sim[effect_cols].sum().round(1)
print(f"Energy over 21 days (kWh): HVAC trimmed={tot.saved_hvac} | lighting dimmed={tot.saved_light} | lab equipment shifted={tot.shifted_lab}")

compare = sim.loc[sim.actual_kw > THRESHOLD_KW, ["actual_kw", "new_kw"]].round(1)
compare



Hours above 40 kW: before=18  after=4
Highest load: before=46.1  after=40.9 kW
Energy over 21 days (kWh): HVAC trimmed=48.3 | lighting dimmed=27.9 | lab equipment shifted=34.5


,actual_kw,new_kw
timestamp,,
2026-11-01 09:00:00,41.4,36.6
2026-11-04 15:00:00,40.1,40.1
2026-11-05 09:00:00,42.7,37.7
2026-11-05 14:00:00,40.6,40.6
2026-11-08 08:00:00,41.6,36.7
2026-11-08 09:00:00,42.1,37.5
2026-11-10 14:00:00,43.1,38.2
2026-11-10 15:00:00,42.3,37.7
2026-11-11 14:00:00,41.4,37.2


In [10]:
# Forecast the day AFTER the last day in the data (real forward-looking forecast)
last_ts = df.timestamp.max()
target_day = (last_ts + pd.Timedelta(hours=1)).normalize()
print("Forecasting:", target_day.date(), target_day.day_name())

base_cols = ["timestamp", "room_id", "building_id", "room_type", "capacity", "day_type",
             "outside_temp_c", "scheduled", TARGET]
hist = df[base_cols].copy()
for c in CATS:
    hist[c] = hist[c].astype(str)

future = (hist[["room_id", "building_id", "room_type", "capacity"]].drop_duplicates("room_id")
          .merge(pd.DataFrame({"timestamp": pd.date_range(target_day, periods=24, freq="h")}), how="cross"))

# ASSUMPTIONS for the target day (replace with the real timetable / weather forecast when available):
#  - timetable = same weekday last week
#  - day type  = same weekday last week
#  - temperature = average of the same hour over the last 3 days
sched_map = hist.set_index(["room_id", "timestamp"])["scheduled"]
temp_map = hist.groupby("timestamp").outside_temp_c.first()
type_map = hist.groupby("timestamp").day_type.first()
week_ago = future.timestamp - pd.Timedelta(days=7)
future["scheduled"] = [sched_map.get((r, t), 0) for r, t in zip(future.room_id, week_ago)]
future["day_type"] = [type_map.get(t, "regular") for t in week_ago]
future["outside_temp_c"] = [np.nanmean([temp_map.get(t - pd.Timedelta(days=k), np.nan) for k in (1, 2, 3)])
                            for t in future.timestamp]

# Same features as before, computed on history + the 24 new hours
full = (pd.concat([hist, future], ignore_index=True)
          .sort_values(["room_id", "timestamp"]).reset_index(drop=True))
g = full.groupby("room_id")[TARGET]
full["lag_24"] = g.shift(24)
full["lag_168"] = g.shift(168)
full["lag_336"] = g.shift(336)
full["same_hour_mean_7d"] = sum(g.shift(24 * k) for k in range(1, 8)) / 7
full["mean_prev_24h"] = g.transform(lambda s: s.shift(24).rolling(24).mean())
full["scheduled"] = full["scheduled"].astype(int)
full["sched_lag_168"] = full.groupby("room_id")["scheduled"].shift(168)
full["hour"] = full.timestamp.dt.hour
full["dow"] = full.timestamp.dt.dayofweek
for c, cats in CATS.items():
    full[c] = pd.Categorical(full[c], categories=cats)

nxt = full[full.timestamp >= target_day].copy()
print("Missing feature values:", int(nxt[FEATURES].isna().sum().sum()))
nxt["pred_kw"] = model.predict(nxt[FEATURES])

fc = nxt.groupby("timestamp").pred_kw.sum().reset_index()
fc["prob_exceed"] = [(p * ratio > THRESHOLD_KW).mean() if p > 15 else 0.0 for p in fc.pred_kw]
top = fc.nlargest(3, "pred_kw").sort_values("timestamp")

print(f"\nPeak forecast for {target_day.date()}: {fc.pred_kw.max():.1f} kW")
print("Peak window:", ", ".join(top.timestamp.dt.strftime("%H:%M")))
print(f"Probability that the day exceeds {THRESHOLD_KW} kW: {1 - np.prod(1 - fc.prob_exceed):.0%}")
print("Hours with probability >= 50%:", ", ".join(fc.loc[fc.prob_exceed >= 0.5, "timestamp"].dt.strftime("%H:%M")) or "none")

fc[fc.pred_kw > 15].round(2)

Forecasting: 2026-11-22 Sunday
Missing feature values: 0

Peak forecast for 2026-11-22: 43.3 kW
Peak window: 08:00, 09:00, 11:00
Probability that the day exceeds 40 kW: 99%
Hours with probability >= 50%: 08:00, 09:00


,timestamp,pred_kw,prob_exceed
8,2026-11-22 08:00:00,42.82,0.91
9,2026-11-22 09:00:00,43.32,0.93
10,2026-11-22 10:00:00,33.79,0.00
11,2026-11-22 11:00:00,36.04,0.00
12,2026-11-22 12:00:00,30.61,0.00
13,2026-11-22 13:00:00,32.62,0.00
14,2026-11-22 14:00:00,30.13,0.00
15,2026-11-22 15:00:00,30.03,0.00
16,2026-11-22 16:00:00,22.43,0.00


In [11]:
import json
os.makedirs("outputs", exist_ok=True)

# 1) Alert configuration (the API will read this later)
config = {
    "name": "Peak alerts v1 (simulated data)",
    "forecast_model": "models/forecast_lgbm.joblib",
    "threshold_kw": THRESHOLD_KW,
    "alert_probability": 0.5,
    "peak_window_hours": 3,
    "min_busy_load_kw": 15,
    "calibration_period": ["2026-10-18", "2026-10-31"],
    "calibration_ratios": [round(float(x), 4) for x in ratio],
    "what_if_rates": RATES,
    "next_day_assumptions": "timetable and day type = same weekday last week; "
                            "temperature = mean of the same hour over the last 3 days",
}
with open("outputs/peak_alert_config.json", "w") as f:
    json.dump(config, f, indent=2)

# 2) Evaluation metrics (for the dashboard / report)
metrics = {
    "evaluation_period": "2026-11-01 to 2026-11-21 (peak value/window metrics: 2026-10-18 to 2026-11-21)",
    "working_days_evaluated": int(len(work)),
    "peak_value_mae_kw": round(float(work.error_kw.abs().mean()), 3),
    "peak_value_error_pct": round(float((work.error_kw.abs() / work.actual_peak_kw).mean() * 100), 2),
    "exact_peak_hour_pct": round(float((work.hour_diff == 0).mean() * 100), 1),
    "within_1_hour_pct": round(float((work.hour_diff <= 1).mean() * 100), 1),
    "top3_window_hit_pct": round(float(work.hit_top3_window.mean() * 100), 1),
    "threshold_kw": THRESHOLD_KW,
    "hourly_alert_precision": round(tp / max(tp + fp, 1), 3),
    "hourly_alert_recall": round(tp / max(tp + fn, 1), 3),
    "what_if_hours_above_threshold_before": before,
    "what_if_hours_above_threshold_after": after,
    "what_if_highest_load_before_kw": round(float(sim.actual_kw.max()), 1),
    "what_if_highest_load_after_kw": round(float(sim.new_kw.max()), 1),
    "note": "Simulated data. What-if rates are assumptions, not measurements.",
}
with open("outputs/peak_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# 3) Tables
summary.to_csv("outputs/peak_alerts_summary.csv", index=False)
fc.round(3).to_csv("outputs/peak_next_day_forecast.csv", index=False)
contrib.to_csv("outputs/peak_contributors.csv", index=False)
compare.reset_index().to_csv("outputs/peak_what_if.csv", index=False)

for name in sorted(os.listdir("outputs")):
    print(f"{name:32s} {os.path.getsize('outputs/' + name):>8,} bytes")

!zip -q -r peak_outputs.zip outputs
files.download("peak_outputs.zip")

peak_alert_config.json              1,569 bytes
peak_alerts_summary.csv             1,169 bytes
peak_contributors.csv                 235 bytes
peak_metrics.json                     633 bytes
peak_next_day_forecast.csv            758 bytes
peak_what_if.csv                      567 bytes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>